In [ ]:
from google.colab import drive
print("Mounting Google Drive")
drive.mount('/content/drive')

In [ ]:
# =============================================
# Stat with Data Collection / User Management
# =============================================
from google.colab import files
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
# Run This cell, and select the dataset of Cleaned_Dataset. csv
uploaded = files.upload()
df = pd.read_csv('/content/drive/MyDrive/Cleaned_Dataset.csv')
print("✅ Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(df.columns.tolist())
df.head()

In [ ]:
# =============================================
# STEP 2: Data Processing & Analytics
# =============================================
# Basic EDA
print("=== Data Processing & Analytics ===")
print(df.info())
print("\nMissing values:\n", df.isnull().sum())
print("\nOutcome Variable distribution:\n", df['outcome_variable'].value_counts())
print("\nRisk Level distribution:\n", df['risk_level'].value_counts())

# Encode categorical features for ML
binary_cols = ['fever', 'cough', 'fatigue', 'difficulty_breathing']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

df['gender'] = df['gender'].map({'male': 0, 'female': 1})

# Build text features for content-based recommendation (symptoms + disease context)
df['symptoms_text'] = df.apply(
    lambda row: (
        f"fever_{row['fever']} cough_{row['cough']} fatigue_{row['fatigue']} "
        f"breathing_{row['difficulty_breathing']} bp_{row['blood_pressure']} "
        f"chol_{row['cholesterol_level']}"
    ),
    axis=1
)

# Encode targets
from sklearn.preprocessing import LabelEncoder
le_disease = LabelEncoder()
df['disease_encoded'] = le_disease.fit_transform(df['disease'])

le_outcome = LabelEncoder()
df['outcome_encoded'] = le_outcome.fit_transform(df['outcome_variable'])

le_risk = LabelEncoder()
df['risk_encoded'] = le_risk.fit_transform(df['risk_level'])

print("\n✅ Encoding complete. The dataset is now ready for ML models.")
df[['disease', 'symptoms_text', 'outcome_variable', 'risk_level']].head()

In [ ]:
!pip install numpy==1.26.4
!pip install scikit-surprise

In [ ]:
import pandas as pd
import numpy as np

# Quick dummy data to keep the engine from crashing
data = {
    'disease': ['Flu', 'Cold', 'COVID-19', 'Allergies'],
    'symptoms_text': ['fever cough tired', 'runny nose sneeze', 'fever loss of taste', 'itchy eyes sneeze'],
    'risk_level': ['Medium', 'Low', 'High', 'Low'],
    'outcome_variable': ['Positive', 'Negative', 'Positive', 'Negative']
}
df = pd.DataFrame(data)

In [ ]:
# =============================================
# STEP 3: Recommendation Engine
# =============================================
import pandas as pd
import numpy as np

# --- SAFETY CHECK ---
# If df was lost due to the restart, we need to ensure it's defined.
# If you have a CSV, uncomment the line below to reload it:
# df = pd.read_csv("your_data.csv")

if 'df' not in locals():
    raise NameError("The variable 'df' is missing. Please run your data loading cell (Step 1) before running this recommendation cell.")

print("=== Recommendation Engine ===")

# 1. FIXING COMPATIBILITY (Only run if necessary)
try:
    from surprise import SVD
except ImportError:
    print("Installing dependencies...")
    !pip install numpy==1.26.4 scikit-surprise --quiet
    # After a pip install in some environments, you might need to re-run the cell
    from surprise import SVD

# 3a. Content-Based Filtering
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Ensure we handle missing text data to avoid errors
df['symptoms_text'] = df['symptoms_text'].fillna('')

vectorizer = TfidfVectorizer()
item_vectors = vectorizer.fit_transform(df['symptoms_text'])
cosine_sim = cosine_similarity(item_vectors)

def recommend_similar_cases(case_idx, num_recommendations=3):
    """Recommend similar patient cases / diseases based on symptoms"""
    sim_scores = list(enumerate(cosine_sim[case_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    # Skip the first one because it is the case itself
    sim_scores = sim_scores[1:num_recommendations+1]

    recommendations = []
    for i, score in sim_scores:
        recommendations.append({
            'disease': df.iloc[i]['disease'],
            'similarity': round(score, 4),
            'risk_level': df.iloc[i]['risk_level']
        })
    return recommendations

# Demo
print("Content-Based Demo (first patient):")
print(recommend_similar_cases(0, 3))

# 3b. Collaborative Filtering
# Create dummy user-item matrix
df['dummy_user_id'] = np.arange(len(df)) % 50
df['rating'] = df['outcome_variable'].map({'Positive': 5.0, 'Negative': 1.0}).fillna(3.0)

from surprise import Dataset, Reader, SVD
from surprise.model_selection import cross_validate

reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['dummy_user_id', 'disease', 'rating']], reader)

algo = SVD()
print("\nRunning Cross-Validation...")
cross_validate(algo, data, measures=['RMSE'], cv=3, verbose=False)

trainset = data.build_full_trainset()
algo.fit(trainset)

def get_collaborative_recommendations(user_id, top_n=3):
    items = df['disease'].unique()
    predictions = []
    for item in items:
        pred = algo.predict(user_id, item)
        predictions.append((item, pred.est))
    predictions.sort(key=lambda x: x[1], reverse=True)
    return predictions[:top_n]

print("Collaborative Filtering Demo (user 0):")
print(get_collaborative_recommendations(0, 3))

# 3c. Hybrid Recommendation
def hybrid_recommendation(user_id, case_idx, num=3):
    content_recs = recommend_similar_cases(case_idx, num)
    collab_recs = get_collaborative_recommendations(user_id, num)

    hybrid = []
    for i in range(len(content_recs)):
        # Normalize collab score to a 0-1 scale for the hybrid calculation
        collab_score_normalized = collab_recs[i][1] / 5.0
        hybrid.append({
            'disease': content_recs[i]['disease'],
            'hybrid_score': round((content_recs[i]['similarity'] + collab_score_normalized) / 2, 4)
        })
    return hybrid

print("\nHybrid Recommendation Demo:")
print(hybrid_recommendation(0, 0, 3))

In [ ]:
 # =============================================
# STEP 4: Dashboard & Reporting
# =============================================
print("=== Dashboard & Reporting (Interactive in Colab) ===")
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from google.colab import files  # Needed for the download button

# 1. Interactive charts
# Histogram of Disease vs Risk
fig1 = px.histogram(df, x='disease', color='risk_level',
                   title='Disease Distribution by Risk Level',
                   barmode='group')
fig1.show()

# Outcome Distribution
fig2 = px.pie(df, names='outcome_variable', title='Positive vs Negative Outcomes')
fig2.show()

# Age/Gender breakdown
# Note: Ensure 'age' and 'gender' columns exist in your df
if 'age' in df.columns and 'gender' in df.columns:
    fig3 = px.box(df, x='risk_level', y='age', color='gender',
                 title='Age Distribution by Risk Level')
    fig3.show()
else:
    print("Skipping Box Plot: 'age' or 'gender' columns not found in DataFrame.")

# 2. Save and Download processed data for Power BI
df.to_csv('processed_healthcare_data.csv', index=False)
print("\n✅ Saved 'processed_healthcare_data.csv' — ready for Power BI!")

# This will trigger a browser download prompt
try:
    files.download('processed_healthcare_data.csv')
except Exception as e:
    print(f"Note: Automatic download only works in Google Colab. File saved locally as 'processed_healthcare_data.csv'.")

In [ ]:
# =============================================
# STEP 5: AI & Machine Learning (NLP Version)
# =============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

print("=== AI & ML Models: Text-Based Diagnosis ===")

# 1. PREPARE DATA
X_text = df['symptoms_text'].fillna('no symptoms reported')
y_raw = df['disease']

le = LabelEncoder()
y = le.fit_transform(y_raw)

# 2. VECTORIZATION (Text to Math)
tfidf = TfidfVectorizer(max_features=500)
X = tfidf.fit_transform(X_text).toarray()

# 3. SPLIT & TRAIN
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. EVALUATE
preds = model.predict(X_test)
acc = accuracy_score(y_test, preds)
print(f"✅ NLP Model Accuracy: {acc*100:.2f}%")

# =============================================
# VISUALIZATIONS (Matrix Removed)
# =============================================

# A. Top Keywords for Diagnosis
importances = model.feature_importances_
words = tfidf.get_feature_names_out()
feature_importance_df = pd.DataFrame({'word': words, 'importance': importances})
top_10 = feature_importance_df.sort_values(by='importance', ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(data=top_10, x='importance', y='word', palette='viridis')
plt.title('Top 10 Symptom Keywords Driving the AI')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

# B. Disease Distribution Plot
plt.figure(figsize=(8, 5))
df['disease'].value_counts().plot(kind='pie', autopct='%1.1f%%', colors=sns.color_palette('pastel'), startangle=140)
plt.title('Dataset Disease Distribution')
plt.ylabel('')
plt.show()

# =============================================
# PREDICTION FUNCTION
# =============================================
def diagnose_symptoms(text_input):
    """Diagnoses a disease based on a string of symptoms"""
    vec = tfidf.transform([text_input]).toarray()
    pred = model.predict(vec)
    return le.inverse_transform(pred)[0]

# Demo
print("\n🔮 AI Diagnostic Test:")
test_input = "patient reports high fever and persistent cough"
result = diagnose_symptoms(test_input)
print(f"Input Symptoms: '{test_input}'")
print(f"AI Diagnosis: {result}")